# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

In [1]:
# !pip install vllm==0.9.0 --quiet

### Comment Out the cell below after first installation.

In [2]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

### Run the cell below every time to activate the installed environment. 

In [3]:
# activate venv after installation. This needs to be run everytime.
# !source ./.venv/bin/activate

/usr/bin/sh: 1: source: not found


## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [1]:
# Added this cell to verify torch is correct before getting too far in.
import os, sys, torch, transformers, vllm, flashinfer

print(sys.executable)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("CUDA_HOME:", os.environ.get("CUDA_HOME"))
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
print("device:", torch.cuda.get_device_name(0))
print("transformers:", transformers.__version__)
print("vllm:", vllm.__version__)
print("flashinfer:", flashinfer.__version__)

INFO 05-03 13:57:03 [__init__.py:243] Automatically detected platform cuda.
/opt/venv/bin/python
CUDA_VISIBLE_DEVICES: 0
CUDA_HOME: /usr/local/cuda
torch: 2.7.0+cu126
torch cuda: 12.6
cuda available: True
device count: 1
device: NVIDIA GeForce RTX 4090
transformers: 4.53.3
vllm: 0.9.0
flashinfer: 0.6.9


In [2]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [3]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [4]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,    
    gpu_memory_utilization=0.90,   # Turned this up from default to util gpu more, after 90% starts sharing RAM
    max_model_len=8192, 
    trust_remote_code=True,
    max_num_seqs=16,            # I think 16 is optimal, going pretty fast seems stable, can try 24/36 but it may slow down
    max_num_batched_tokens=8192, 
    # enforce_eager=True, # This was killing the gpu/cpu 
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.0, # greedy
    top_p=1.0,
    top_k=-1,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)
 
print("Model loaded.")

INFO 05-03 16:25:10 [config.py:793] This model supports multiple tasks: {'generate', 'score', 'reward', 'classify', 'embed'}. Defaulting to 'generate'.
WARNING 05-03 16:25:10 [config.py:907] bitsandbytes quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 05-03 16:25:10 [config.py:2118] Chunked prefill is enabled with max_num_batched_tokens=8192.


Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/root/.local/share/uv/python/cpython-3.10-linux-x86_64-gnu/lib/python3.10/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/root/.local/share/uv/python/cpython-3.10-linux-x86_64-gnu/lib/python3.10/multiprocessing/spawn.py", line 126, in _main


KeyboardInterrupt: 

In [6]:
import subprocess
print(subprocess.check_output(['nvidia-smi']).decode())

Sun May  3 04:09:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.01              Driver Version: 596.36         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        On  |   00000000:02:00.0  On |                  Off |
|  0%   34C    P2            108W /  599W |   24112MiB /  24564MiB |     22%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [7]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="auto",
# )


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [6]:
# Build prompts for first 5 entries
prompts = []
for item in data:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 1126 questions...


Adding requests:   0%|          | 0/1126 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1126 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…


── Response 0 (id=0) ──
Okay, let's see. I need to find the sum of the first 325 positive even whole numbers. Hmm, first, let me make sure I know what the first few even whole numbers are. Positive even whole numbers start from 2, right? So the first one is 2, the second is 4, the third is 6, and so on. So the nth positive even whole number is 2n. Let me confirm that. For n=1, 2*1=2, which is correct. n=2, 2*2=4, correct ...

── Response 1 (id=1) ──
Okay, let's try to solve this integral: the integral from negative infinity to positive infinity of (a^(3/2)) divided by (s² + a²) ds. Hmm, first, I need to check if the integral converges. Since the integrand is even (because s² is even, and a is a constant), maybe I can compute it from 0 to infinity and double it. But first, let's recall that the integral of 1/(s² + b²) ds from -infty to infty i ...

── Response 2 (id=2) ──
Okay, let's try to solve this problem step by step. First, part (a) is about a turkey cooling down from 185°F to 15

In [11]:
# Save outputs so you don't have to do it again
import json
import os
from datetime import datetime

SAVE_PATH = "qwen_responses_saved.json"

def count_generated_tokens(out):
    completion = out.outputs[0]

    # Exact vLLM generated token count, if available
    if hasattr(completion, "token_ids") and completion.token_ids is not None:
        return len(completion.token_ids)

    # Fallback
    return None


records = []

for i, out in enumerate(outputs):
    completion = out.outputs[0]
    response_text = completion.text.strip()

    record = {
        "index": i,
        "id": data[i].get("id"),
        "question": data[i].get("question"),
        "options": data[i].get("options"),
        "response": response_text,
        "generated_tokens": count_generated_tokens(out),
        "finish_reason": getattr(completion, "finish_reason", None),
    }

    records.append(record)


payload = {
    "saved_at": datetime.now().isoformat(),
    "num_records": len(records),
    "records": records,
}


# Atomic save: write temp file first, then replace final file
tmp_path = SAVE_PATH + ".tmp"

with open(tmp_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

os.replace(tmp_path, SAVE_PATH)

print(f"Saved {len(records)} responses to {SAVE_PATH}")

Saved 1126 responses to qwen_responses_saved.json


In [14]:
# Reload from saved file

import json
import re
import sys
from tqdm.auto import tqdm

SAVE_PATH = "qwen_responses_saved.json"

with open(SAVE_PATH, "r", encoding="utf-8") as f:
    saved = json.load(f)

records = saved["records"]

print(f"Loaded {len(records)} saved responses from {SAVE_PATH}")
print(f"Expected {len(data)} data items")

assert len(records) == len(data), (
    f"Mismatch: saved has {len(records)} responses, data has {len(data)} items"
)

# Recover plain response list in original order
responses = [r["response"] for r in records]

Loaded 1126 saved responses from qwen_responses_saved.json
Expected 1126 data items


In [8]:
import torch
print(torch.cuda.memory_allocated(0) / 1e9, "GB allocated")
print(torch.cuda.get_device_name(0))

0.0 GB allocated
NVIDIA GeForce RTX 4090


### Generate with Transformers (for Datahub)

In [21]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Tokenize (padded batch)
# print(f"Generating responses for {len(prompts)} questions...")
# inputs = tokenizer(
#     prompts,
#     return_tensors="pt",
#     padding=True,
#     truncation=True,
#     max_length=16384,
# ).to(llm.device)

# # Generate
# with torch.no_grad():
#     output_ids = llm.generate(
#         **inputs,
#         max_new_tokens=MAX_TOKENS,
#         temperature=0.6,
#         top_p=0.95,
#         top_k=20,
#         repetition_penalty=1.0,
#         do_sample=True,
#     )

# # Decode only the new tokens (strip the prompt)
# responses = []
# for i, out in enumerate(output_ids):
#     new_tokens = out[inputs["input_ids"].shape[1]:]
#     responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [15]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring:   0%|          | 0/1126 [00:00<?, ?it/s]

Scoring complete. 1126 results.


## 8. Summary

Print accuracy broken down by question type.

In [16]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :  180 /  375  (48.00%)
  Free-form  :  399 /  751  (53.13%)
  Overall    :  579 / 1126  (51.42%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [17]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 1126 records to results/starter_results.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!

In [20]:
# Saved statistics 
import json
import statistics

SAVE_PATH = "qwen_responses_saved.json"

with open(SAVE_PATH, "r", encoding="utf-8") as f:
    saved = json.load(f)

records = saved["records"]

# Pull generated token counts
token_counts = [r.get("generated_tokens") for r in records]

# If any are None, recompute from response text
if any(t is None for t in token_counts):
    print("Some generated_tokens are missing. Recomputing from response text...")
    token_counts = [
        len(tokenizer.encode(r["response"], add_special_tokens=False))
        for r in records
    ]

# Basic stats
total_tokens = sum(token_counts)
min_tokens = min(token_counts)
max_tokens = max(token_counts)
mean_tokens = statistics.mean(token_counts)
median_tokens = statistics.median(token_counts)



print("Generated token statistics")
print("--------------------------")
print(f"Num responses: {len(token_counts)}")
print(f"Total generated tokens: {total_tokens}")
print(f"Min: {min_tokens}")
print(f"Max: {max_tokens}")
print(f"Mean: {mean_tokens:.2f}")
print(f"Median: {median_tokens}")
print(f"P90: {p90}")
print(f"P95: {p95}")
print(f"P99: {p99}")

# Find the output(s) with max tokens
max_indices = [i for i, t in enumerate(token_counts) if t == max_tokens]

print("\nMax-token responses:")
for i in max_indices[:10]:
    r = records[i]
    print(f"\nindex={r['index']} id={r.get('id')} generated_tokens={token_counts[i]}")
    print(r["response"][:500], "..." if len(r["response"]) > 500 else "")

Generated token statistics
--------------------------
Num responses: 1126
Total generated tokens: 4802643
Min: 319
Max: 8094
Mean: 4265.22
Median: 3701.0
P90: 7963
P95: 8013
P99: 8059

Max-token responses:

index=27 id=27 generated_tokens=8094
Okay, let's see. I need to find the half-life of an element that decays by 3.416% each day. Hmm, half-life is the time it takes for half of the substance to decay, right? So first, I should remember the formula for exponential decay. 

Exponential decay can be modeled by the equation: N(t) = N0 * (1 - r)^t, where N(t) is the remaining quantity after time t, N0 is the initial quantity, r is the daily decay rate (as a decimal), and t is time in days. 

But wait, the problem says it decays by 3.416 ...
